In [10]:
import pandas as pd
import spotipy

In [11]:
df = pd.read_csv(
    "../pipeline/data/processed/spotify/lastfm_recenttracks.csv",
    encoding="utf-8"
)

In [12]:
df = df[["track_name", "artist_name"]]
df.head()

,track_name,artist_name
0,Professional,The Weeknd
1,Vampiro,Matuê
2,Heartbeat,Childish Gambino
3,When I’m Home,James Blake
4,Both (feat. Drake),Gucci Mane


In [13]:
df = df.drop_duplicates(["track_name", "artist_name"])
df = df.head(10)

In [14]:
df["spotify_id"] = df.apply(
    lambda row: get_track_id(row["track_name"], row["artist_name"], CLIENT_ID, CLIENT_SECRET), axis=1
)

df["isrc"] = df.apply(
    lambda row: get_track_isrc(row["track_name"], row["artist_name"], CLIENT_ID, CLIENT_SECRET), axis=1
)

In [15]:
def get_features_safe(row):
    try:
        return get_track_features(row["isrc"])
    except Exception as e:
        print(
            f"Erro: {row['artist_name']} - "
            f"{row['track_name']} - "
            f"{row['isrc']} | {e}"
        )
        return None


df["features"] = df.apply(
    get_features_safe,
    axis=1
)

Erro: The Weeknd - Professional - USUM71310341 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USUM71310341/features
Erro: Matuê - Vampiro - BXW922100034 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/BXW922100034/features
Erro: Childish Gambino - Heartbeat - USYAH1100351 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USYAH1100351/features
Erro: James Blake - When I’m Home - USQ4E2600373 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USQ4E2600373/features
Erro: Gucci Mane - Both (feat. Drake) - USAT21603373 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USAT21603373/features
Erro: Jhené Aiko - Stay Ready (What a Life) - USUM71312344 | 500 Server Error: Internal Server Error for url: https://melodata1.p.rapidapi.com/tracks/USUM71312344/features
Erro: Frank Ocean - Novacan

In [ ]:
df.to_csv("parte_1.csv", index=False, encoding="utf-8")

In [16]:
import requests

def search_recco(track_name, artist_name):
    url = "https://api.reccobeats.com/v1/track/search"

    params = {
        "searchText": track_name,
        "artist": artist_name
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

def search_recco_safe(row):
    try:
        return search_recco(row["track_name"], row["artist_name"])
    except Exception as e:
        print(
            f"Erro: {row['artist_name']} - "
            f"{row['track_name']} - "
            f"{row['isrc']} | {e}"
        )
        return None

df["recco_result"] = df.apply(
    lambda row: search_recco_safe(row),
    axis=1
)

df.head()

,track_name,artist_name,spotify_id,isrc,features,recco_result
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,{'content': [{'id': 'c9ea3a2d-dc6a-4462-9aa6-5...
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,{'content': [{'id': '8492861c-06b0-44aa-a991-e...
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,{'content': [{'id': '18f5a77a-bdf4-4a11-a3f4-3...
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,"{'content': [], 'page': 0, 'size': 25, 'totalE..."
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,{'content': [{'id': 'f8186f9d-cdac-44f0-86bf-a...


In [17]:
df.iloc[1]["recco_result"]

{'content': [{'id': '8492861c-06b0-44aa-a991-ebeb3d1ef53e',
   'trackTitle': 'Vampiro',
   'artists': [{'id': '3cba4e20-f96a-4aa4-a82f-0afead857a2d',
     'name': 'Matuê',
     'href': 'https://open.spotify.com/artist/5nP8x4uEFjAAmDzwOEc9b8'},
    {'id': '188e6f5d-b392-486b-9b4c-d97cdacc50a3',
     'name': 'WIU',
     'href': 'https://open.spotify.com/artist/3MrDVzg7ZXaYMyQmbDInr7'},
    {'id': '2d6069a8-0f61-4ac0-814e-44c5e45c0fb4',
     'name': 'Teto',
     'href': 'https://open.spotify.com/artist/68YeXpLt3jB7JHQS5ZjMGo'}],
   'durationMs': 250434,
   'isrc': 'BXW922100034',
   'ean': None,
   'upc': None,
   'href': 'https://open.spotify.com/track/6bTdZ7xfKp3NqqADJ8HLyj',
   'availableCountries': 'AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,DE,EC,EE,SV,FI,FR,GR,GT,HN,HK,HU,IS,IE,IT,LV,LT,LU,MY,MT,MX,NL,NZ,NI,NO,PA,PY,PE,PH,PL,PT,SG,SK,ES,SE,CH,TW,TR,UY,US,GB,AD,LI,MC,ID,JP,TH,VN,RO,IL,ZA,SA,AE,BH,QA,OM,KW,EG,MA,DZ,TN,LB,JO,PS,IN,BY,KZ,MD,UA,AL,BA,HR,ME,MK,RS,SI,KR,BD,PK,LK,GH,KE,NG

In [18]:
df["recco_id"] = df["recco_result"].apply(
    lambda x: x["content"][0]["id"] if x["content"] else None
)

df.drop(["recco_result"], axis=1, inplace=True)
df

,track_name,artist_name,spotify_id,isrc,features,recco_id
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None


In [19]:
import requests


def get_recco_audio_features(track_id):
    url = f"https://api.reccobeats.com/v1/track/{track_id}/audio-features"

    headers = {
        "Accept": "application/json"
    }

    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [20]:
df["features_recco"] = df["recco_id"].apply(
    lambda x: get_recco_audio_features(x) if x else None)

df

,track_name,artist_name,spotify_id,isrc,features,recco_id,features_recco
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,"{'id': 'c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b',..."
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e,"{'id': '8492861c-06b0-44aa-a991-ebeb3d1ef53e',..."
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,"{'id': '18f5a77a-bdf4-4a11-a3f4-3e4b570cb814',..."
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None,None
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753,"{'id': 'f8186f9d-cdac-44f0-86bf-aa18ca19a753',..."
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535,"{'id': 'f1920772-e80a-488b-833c-04b4f2956535',..."
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,"{'id': 'f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1',..."
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866,"{'id': '65c51efa-45e3-415c-b770-c7f4a2f45866',..."
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,"{'id': 'e52ef438-7cf1-47ac-bb60-f800b52ea3d6',..."
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None,None


In [21]:
df_features = df.copy()[["track_name", "features", "features_recco"]]

In [22]:
df_features.iloc[0]["features_recco"]

{'id': 'c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b',
 'href': 'https://open.spotify.com/track/5ZicFGBDAi9J2YCVesboUp',
 'isrc': 'USUM71310341',
 'acousticness': 0.163,
 'danceability': 0.405,
 'energy': 0.619,
 'instrumentalness': 0.000322,
 'key': 11,
 'liveness': 0.0788,
 'loudness': -8.92,
 'mode': 0,
 'speechiness': 0.0618,
 'tempo': 120.087,
 'valence': 0.234}

In [23]:
features = pd.json_normalize(df["features_recco"])

df = pd.concat(
    [df.drop(columns=["features_recco"]), features],
    axis=1
)

df

,track_name,artist_name,spotify_id,isrc,features,recco_id,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,Professional,The Weeknd,5ZicFGBDAi9J2YCVesboUp,USUM71310341,None,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,c9ea3a2d-dc6a-4462-9aa6-56ff93f3b60b,https://open.spotify.com/track/5ZicFGBDAi9J2YC...,USUM71310341,0.16300,0.405,0.619,0.000322,11.0,0.0788,-8.920,0.0,0.0618,120.087,0.234
1,Vampiro,Matuê,6bTdZ7xfKp3NqqADJ8HLyj,BXW922100034,None,8492861c-06b0-44aa-a991-ebeb3d1ef53e,8492861c-06b0-44aa-a991-ebeb3d1ef53e,https://open.spotify.com/track/6bTdZ7xfKp3NqqA...,BXW922100034,0.01360,0.782,0.643,0.000000,8.0,0.0654,-4.956,0.0,0.0438,114.994,0.627
2,Heartbeat,Childish Gambino,5AGQSF0ytihJyt96K5vW9d,USYAH1100351,None,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,18f5a77a-bdf4-4a11-a3f4-3e4b570cb814,https://open.spotify.com/track/5AGQSF0ytihJyt9...,USYAH1100351,0.00373,0.800,0.545,0.000511,1.0,0.0445,-7.002,0.0,0.1270,119.959,0.291
3,When I’m Home,James Blake,5nigMKgXqeNvaiqEBWwo9s,USQ4E2600373,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Both (feat. Drake),Gucci Mane,5tFep7dXGd7vEJ668wTPux,USAT21603373,None,f8186f9d-cdac-44f0-86bf-aa18ca19a753,f8186f9d-cdac-44f0-86bf-aa18ca19a753,https://open.spotify.com/track/0iQXqF8CYMI7mLN...,USAT21603373,0.11900,0.849,0.405,0.000118,7.0,0.0707,-7.509,0.0,0.2260,139.976,0.344
5,Stay Ready (What a Life),Jhené Aiko,5nkUIVKqOqdpB6ApKgEMkv,USUM71312344,None,f1920772-e80a-488b-833c-04b4f2956535,f1920772-e80a-488b-833c-04b4f2956535,https://open.spotify.com/track/5nkUIVKqOqdpB6A...,USUM71312344,0.45500,0.347,0.493,0.003660,8.0,0.1270,-11.548,0.0,0.2900,82.887,0.308
6,Jungle,Drake,7JXZq0JgG2zTrSOAgY8VMC,USCM51500037,"{'data': {'isrc': 'USCM51500037', 'title': 'Ju...",f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,f30a1ce4-7adb-4ca7-bdc7-9f70110f90d1,https://open.spotify.com/track/0mnABjjVAEmnSso...,USCM51500038,0.61800,0.685,0.226,0.000229,7.0,0.1070,-8.694,1.0,0.0466,99.786,0.418
8,Novacane,Frank Ocean,4osgfFTICMkcGbbigdsa53,USUM71107257,None,65c51efa-45e3-415c-b770-c7f4a2f45866,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,https://open.spotify.com/track/53qYItjefG5SUf6...,USUM72111952,0.12900,0.654,0.383,0.000755,3.0,0.1030,-9.875,0.0,0.0363,139.950,0.268
9,The Zone,The Weeknd,53qYItjefG5SUf62428dIw,USUM72111952,None,e52ef438-7cf1-47ac-bb60-f800b52ea3d6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Lose You,Drake,2Na0z2gfN67Rzf0vp74Wi3,USCM51700071,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df.to_csv("../pipeline/data/processed/spotify/enriched_data.csv", index=False, encoding="utf-8")

In [2]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


def get_track_isrc(song_name, artist_name, client_id, client_secret):
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"track:{song_name} artist:{artist_name}"

    results = sp.search(
        q=query,
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if not tracks:
        return None

    return tracks[0]["external_ids"].get("isrc")

In [3]:
def get_track_id(song_name, artist_name, client_id, client_secret):
    sp = spotipy.Spotify(
        auth_manager=SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
    )

    query = f"track:{song_name} artist:{artist_name}"

    results = sp.search(
        q=query,
        type="track",
        limit=1
    )

    tracks = results["tracks"]["items"]

    if not tracks:
        return None

    return tracks[0]["id"]

In [4]:
CLIENT_ID = "d61d466b88404563a016f197e8572c5e"
CLIENT_SECRET = "760825fb70dc4d449ad30950e90e904d"

id = get_track_id(
    "Blinding Lights",
    "The Weeknd",
    CLIENT_ID,
    CLIENT_SECRET
)

id

'0VjIjW4GlUZAMYd2vXMi3b'

In [5]:
CLIENT_ID = "d61d466b88404563a016f197e8572c5e"
CLIENT_SECRET = "760825fb70dc4d449ad30950e90e904d"

isrc = get_track_isrc(
    "jungle",
    "Drake",
    CLIENT_ID,
    CLIENT_SECRET
)

isrc

'USCM51500037'

In [6]:
import requests


def search_track(song_name, limit=10, offset=0):

    url = "https://melodata1.p.rapidapi.com/tracks/search"

    params = {
        "q": song_name,
        "offset": offset,
        "limit": limit
    }

    headers = {
        "Content-Type": "application/json",
        "x-rapidapi-host": "melodata1.p.rapidapi.com",
        "x-rapidapi-key": "dbcd34ab68mshbdcca93581f9718p1c56b8jsn2fe95019b903"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [7]:
data = search_track("Professional")

data

{'data': {'results': [{'isrc': 'GBAYE2200698',
    'title': 'Professional',
    'artist': 'Gabriels',
    'album': 'Angels & Queens (Deluxe)',
    'release_date': '2023-07-07T00:00:00.000Z'},
   {'isrc': 'USNA10319594',
    'title': 'Professional Daydreamer',
    'artist': 'Over the Rhine',
    'album': 'Ohio',
    'release_date': None},
   {'isrc': 'USAT29900495',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Boys for Pele',
    'release_date': None},
   {'isrc': 'USAT20625633',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Boys for Pele',
    'release_date': None},
   {'isrc': 'USRH11604040',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Tales of a Librarian',
    'release_date': None},
   {'isrc': 'USAT20619231',
    'title': 'Professional Widow',
    'artist': 'Tori Amos',
    'album': 'Tales of a Librarian',
    'release_date': None},
   {'isrc': 'USRH11604343',
    'title': 'Professional Wid

In [8]:
import requests


def get_track_features(isrc):
    url = f"https://melodata1.p.rapidapi.com/tracks/{isrc}/features"

    headers = {
        "Content-Type": "application/json",
        "x-rapidapi-host": "melodata1.p.rapidapi.com",
        "x-rapidapi-key": "dbcd34ab68mshbdcca93581f9718p1c56b8jsn2fe95019b903"
    }

    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [9]:
features = get_track_features(
    "USUG11904206"
)

features

{'data': {'isrc': 'USUG11904206',
  'title': 'Blinding Lights',
  'artist': 'The Weeknd',
  'features': {'bpm': 170.8,
   'key': 'Fm',
   'key_confidence': 0.8761205,
   'energy': 0.7539831,
   'danceability': 0.4590787,
   'valence': None,
   'acousticness': None,
   'loudness': -8.5,
   'instrumentalness': None,
   'speechiness': 0.1168,
   'liveness': None,
   'time_signature': 4},
  'analysis_version': '1.2',
  'source': 'essentia'},
 'meta': {'quota': {'used': 0,
   'limit': None,
   'resets_at': '2026-09-13T14:04:40.981Z'},
  'request_id': 'req_8046ae065c1b'}}